In [1]:
import pandas as pd
import numpy as np
import altair as alt
import theme

alt.themes.register('main_theme', theme.main_theme)
alt.themes.enable('main_theme')

alt.data_transformers.disable_max_rows()

/tmp/ipykernel_25650/1193958727.py:6: AltairDeprecationWarning: 
Deprecated since `altair=5.5.0`. Use altair.theme instead.
Most cases require only the following change:

    # Deprecated
    alt.themes.enable('quartz')

    # Updated
    alt.theme.enable('quartz')

If your code registers a theme, make the following change:

    # Deprecated
    def custom_theme():
        return {'height': 400, 'width': 700}
    alt.themes.register('theme_name', custom_theme)
    alt.themes.enable('theme_name')

    # Updated
    @alt.theme.register('theme_name', enable=True)
    def custom_theme():
        return alt.theme.ThemeConfig(
            {'height': 400, 'width': 700}
        )

See the updated User Guide for further details:
    https://altair-viz.github.io/user_guide/api.html#theme
    https://altair-viz.github.io/user_guide/customization.html#chart-themes
  alt.themes.register('main_theme', theme.main_theme)


DataTransformerRegistry.enable('default')

In [2]:
# Load your dataset
func_data = pd.read_csv('293T-TIM1_entry_func_effects.csv')
func_data.head()

,site,wildtype,mutant,effect,effect_std,times_seen,n_selections
0,26,Q,*,-3.3850,0.0000,20.670,6
1,26,Q,-,-1.5110,0.5134,0.500,3
2,26,Q,A,-0.4125,0.6007,5.333,6
3,26,Q,C,-0.9769,0.3134,5.667,6
4,26,Q,D,-0.4456,0.1056,5.333,6


In [3]:
# read in structure mapping
site_map = pd.read_csv('site_numbering_map.csv')
site_map.head()

,wildtype,sequential_site,reference_site,region
0,Q,1,26,epitope-1
1,N,2,27,epitope-1
2,I,3,28,epitope-1
3,T,4,29,epitope-1
4,E,5,30,epitope-1


In [4]:
func_data_for_heatmap = pd.merge(
    func_data,
    site_map[['reference_site', 'region']],
    left_on='site',
    right_on='reference_site',
    how='right'
).drop(
    columns=['reference_site']
).query(
    'times_seen >= 2'
).query(
    'effect_std <= 2'
)

func_data_for_heatmap.to_csv('filtered_cell_entry.csv', index=False)


In [5]:
func_data_for_heatmap[['region', 'effect']].dropna().groupby('region').size()



region
epitope-0    1274
epitope-1     752
epitope-2     451
epitope-3     780
epitope-4     937
epitope-5    1324
dtype: int64

In [6]:
# ---------------------------------------------------------------------------
# Define colors and order
order = [
    'epitope-0',
    'epitope-1',
    'epitope-2',
    'epitope-3',
    'epitope-4',
    'epitope-5'
]

colors = {
    'epitope-0': '#F28C8C',   # pink
    'epitope-5': '#F7D593',   # peach
    'epitope-3': '#B7E1CD',   # mint green
    'epitope-4': '#C4B4E7',   # lavender
    'epitope-2': '#FDF699',   # light yellow
    'epitope-1': '#AFCDE7'    # light blue
}

# ---------------------------------------------------------------------------
# Filter to only rows with valid region and effect
plot_df = func_data_for_heatmap.dropna(
    subset=['region', 'effect']
).copy()

# Add jitter for yOffset (Altair requires this precomputed)
plot_df['jitter'] = np.random.normal(
    loc=0,
    scale=0.2,
    size=len(plot_df)
)

# ---------------------------------------------------------------------------
# Scatter points
points = alt.Chart(plot_df).mark_circle(
    size=40,
    opacity=0.3
).encode(
    y=alt.Y(
        "region:N",
        sort=order,
        title=None
    ),
    x=alt.X(
        "effect:Q",
        title=["Mutation effect on", "cell entry"]
    ),
    yOffset=alt.YOffset("jitter:Q"),
    color=alt.Color(
        "region:N",
        scale=alt.Scale(
            domain=order,
            range=[colors[o] for o in order]
        ),
        legend=None
    ),
    tooltip=['site', 'wildtype', 'mutant', 'effect']
)

# ---------------------------------------------------------------------------
# Median effect lines
median_line = alt.Chart(plot_df).mark_tick(
    color='black',
    thickness=3,
    size=25
).encode(
    y=alt.Y(
        'region:N',
        sort=order
    ),
    x='median(effect):Q'
)

# ---------------------------------------------------------------------------
# Vertical reference line at x = 0
vline = alt.Chart(
    pd.DataFrame({'x': [0]})
).mark_rule(
    color='black',
    size=1.25,
    opacity=1.0,
    strokeDash=[6, 6]
).encode(
    x='x:Q'
)

# ---------------------------------------------------------------------------
# Combine all layers
chart = alt.layer(
    points,
    median_line,
    vline
).properties(
    height=200,
    width=300,
    title=alt.TitleParams(
        text='Antigenic region',
        anchor='middle',
        fontSize=16,
        fontWeight='bold',
    )
)

chart.display()


alt.LayerChart(...)

In [7]:
# ---------------------------------------------------------------------------
# Define colors and order
order = [
    'epitope-0',
    'epitope-1',
    'epitope-2',
    'epitope-3',
    'epitope-4',
    'epitope-5'
]

colors = {
    'epitope-0': '#A9A9A9',   
    'epitope-5': '#A9A9A9',  
    'epitope-3': '#A9A9A9',  
    'epitope-4': '#A9A9A9',  
    'epitope-2': '#A9A9A9',   
    'epitope-1': '#A9A9A9'    
}

# ---------------------------------------------------------------------------
# Filter to only rows with valid region and effect
plot_df = func_data_for_heatmap.dropna(
    subset=['region', 'effect']
).copy()

# Add jitter for yOffset (Altair requires this precomputed)
plot_df['jitter'] = np.random.normal(
    loc=0,
    scale=0.2,
    size=len(plot_df)
)

# ---------------------------------------------------------------------------
# Scatter points
points = alt.Chart(plot_df).mark_circle(
    size=40,
    opacity=0.3
).encode(
    y=alt.Y(
        "region:N",
        sort=order,
        title=None
    ),
    x=alt.X(
        "effect:Q",
        title=["Mutation effect on cell entry"]
    ),
    yOffset=alt.YOffset("jitter:Q"),
    color=alt.Color(
        "region:N",
        scale=alt.Scale(
            domain=order,
            range=[colors[o] for o in order]
        ),
        legend=None
    ),
    tooltip=['site', 'wildtype', 'mutant', 'effect']
)

# ---------------------------------------------------------------------------
# Median effect lines
median_line = alt.Chart(plot_df).mark_tick(
    color='black',
    thickness=3,
    size=25
).encode(
    y=alt.Y(
        'region:N',
        sort=order
    ),
    x='median(effect):Q'
)

# ---------------------------------------------------------------------------
# Vertical reference line at x = 0
vline = alt.Chart(
    pd.DataFrame({'x': [0]})
).mark_rule(
    color='black',
    size=1.25,
    opacity=1.0,
    strokeDash=[6, 6]
).encode(
    x='x:Q'
)

# ---------------------------------------------------------------------------
# Combine all layers
chart = alt.layer(
    points,
    median_line,
    vline
).properties(
    height=200,
    width=300,
    title=alt.TitleParams(
        text='Antigenic region',
        anchor='middle',
        fontSize=16,
        fontWeight='bold',
    )
)

chart.display()


alt.LayerChart(...)